<a href="https://colab.research.google.com/github/EvenSol/NeqSim-Colab/blob/master/notebooks/fluidflow/finite_element_methods_oil_gas_neqsim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finite-element methods for oil & gas engineering with NeqSim

This notebook establishes the reusable open-source stack **NeqSim → Gmsh → scikit-fem / FEniCSx → PyVista**, while retaining OpenFOAM for CFD. NeqSim owns thermodynamics, phase equilibrium and fluid properties; Gmsh owns geometry/meshing; scikit-fem provides lightweight transparent FEM; FEniCSx handles general PDE/multiphysics; PyVista provides a common visualization layer.

Executable examples: (1) radial insulated-pipe heat transfer with scikit-fem, (2) damaged-insulation geometry with Gmsh → FEniCSx → PyVista, (3) NeqSim molecular diffusivity → porous-rock diffusion, and (4) wellbore-to-formation heat conduction with FEniCSx. The companion `neqsim_fenicsx_fem_pipeline.ipynb` adds transient cooldown, hydrate margin and thermo-elastic pipe stress.

In [ ]:
import hashlib, importlib.metadata, os, shutil, subprocess, sys
from pathlib import Path
NEQSIM_SOURCE_REF='master'; os.environ['NEQSIM_JVM_AUTOSTART']='0'
def rq(cmd,cwd=None): return subprocess.run(cmd,cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,check=True).stdout
rq([sys.executable,'-m','pip','install','-q','neqsim','scikit-fem','gmsh','meshio','pyvista','scipy'])
try:
 import dolfinx
except ImportError:
 p=Path('/tmp/fenicsx.sh'); rq(['wget','-q','https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh','-O',str(p)]); rq(['bash',str(p)])
src=Path('/content/neqsim-java')
if src.exists(): shutil.rmtree(src)
rq(['git','clone','--depth','1','--branch',NEQSIM_SOURCE_REF,'https://github.com/equinor/neqsim.git',str(src)]); commit=rq(['git','-C',str(src),'rev-parse','HEAD']).strip(); rq(['./mvnw','-q','-DskipTests','-P','shade','package'],cwd=src); jar=sorted((src/'target').glob('neqsim-*-shaded.jar'))[-1]; sha=hashlib.sha256(jar.read_bytes()).hexdigest()
import jpype
jpype.addClassPath(str(jar))
if not jpype.isJVMStarted(): jpype.startJVM('-Xrs',convertStrings=False,interrupt=False)
SystemSrkEos=jpype.JClass('neqsim.thermo.system.SystemSrkEos'); Ops=jpype.JClass('neqsim.thermodynamicoperations.ThermodynamicOperations'); MainOnly=jpype.JClass('neqsim.pvtsimulation.flowassurance.SurfCooldownAnalyzer'); loc=str(MainOnly.class_.getProtectionDomain().getCodeSource().getLocation()); assert jar.name in loc
print('NeqSim master commit:',commit); print('JAR SHA-256:',sha); print('Main-only source:',loc); print('DOLFINx:',importlib.metadata.version('fenics-dolfinx')); print('scikit-fem:',importlib.metadata.version('scikit-fem')); print('Gmsh:',importlib.metadata.version('gmsh')); print('PyVista:',importlib.metadata.version('pyvista'))

## Common NeqSim state

One gas state is reused so that every FEM example has an explicit thermodynamic handoff rather than arbitrary fluid constants. NeqSim also supplies a multicomponent effective CO2 diffusion coefficient.

In [ ]:
import gmsh, meshio, numpy as np, pandas as pd, matplotlib.pyplot as plt, pyvista as pv, ufl
from skfem import Basis,BilinearForm,FacetBasis,LinearForm,MeshLine,MeshTri,ElementLineP1,ElementTriP1,asm,condense,solve
from skfem.helpers import dot,grad
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem,mesh,plot
from dolfinx.fem.petsc import LinearProblem
from dolfinx.io import gmsh as gmshio
pv.OFF_SCREEN=True
Tg,Pg=45.,75.; comp={'nitrogen':.01,'CO2':.03,'methane':.84,'ethane':.07,'propane':.03,'i-butane':.005,'n-butane':.01,'n-pentane':.005}
f=SystemSrkEos(Tg+273.15,Pg)
for n,x in comp.items(): f.addComponent(n,x)
f.setMixingRule('classic'); Ops(f).TPflash(); f.initPhysicalProperties(); gas=f.getPhase('gas'); pp=gas.getPhysicalProperties(); pp.setDiffusionCoefficientModel('Fuller-Schettler-Giddings'); gas.initPhysicalProperties(); pp.calcEffectiveDiffusionCoefficients()
pr={'rho':float(gas.getDensity('kg/m3')),'mu':float(gas.getViscosity('kg/msec')),'k':float(gas.getThermalConductivity('W/mK')),'cp':float(gas.getCp('J/kgK')),'D_CO2':float(pp.getEffectiveDiffusionCoefficient('CO2'))}; display(pd.DataFrame({'property':pr.keys(),'value':pr.values()}))

## A. scikit-fem — lightweight radial insulated-pipe heat transfer

This is the model to start with when the geometry is uniform. The radial axisymmetric weak form is compared directly with the analytical cylindrical-resistance solution.

In [ ]:
Di,ts,ti=.254,.0127,.05; ri,rs,ro=Di/2,Di/2+ts,Di/2+ts+ti; ks,ki=50.,.17; Tsea,ho,vel=4.,300.,5.
Re=pr['rho']*vel*Di/pr['mu']; Pr=pr['cp']*pr['mu']/pr['k']; fd=(.79*np.log(Re)-1.64)**-2; Nu=(fd/8)*(Re-1000)*Pr/(1+12.7*np.sqrt(fd/8)*(Pr**(2/3)-1)); hi=Nu*pr['k']/Di
m=MeshLine(np.linspace(ri,ro,121)); e=ElementLineP1(); b=Basis(m,e); fi=FacetBasis(m,e,facets=m.facets_satisfying(lambda x:np.isclose(x[0],ri))); fo=FacetBasis(m,e,facets=m.facets_satisfying(lambda x:np.isclose(x[0],ro)))
@BilinearForm
def cnd(u,v,w): return np.where(w.x[0]<=rs,ks,ki)*w.x[0]*dot(grad(u),grad(v))
@BilinearForm
def rbi(u,v,w): return hi*w.x[0]*u*v
@BilinearForm
def rbo(u,v,w): return ho*w.x[0]*u*v
@LinearForm
def li(v,w): return hi*w.x[0]*(Tg+273.15)*v
@LinearForm
def lo(v,w): return ho*w.x[0]*(Tsea+273.15)*v
A=asm(cnd,b)+asm(rbi,fi)+asm(rbo,fo); rhs=asm(li,fi)+asm(lo,fo); Tk=solve(A,rhs)-273.15; rr=m.p[0]
R=1/(hi*2*np.pi*ri)+np.log(rs/ri)/(2*np.pi*ks)+np.log(ro/rs)/(2*np.pi*ki)+1/(ho*2*np.pi*ro); qa=(Tg-Tsea)/R; Ta=Tg-qa/(hi*2*np.pi*ri); Ts=float(Tk[np.argmin(abs(rr-ri))]); print('scikit-fem inner wall:',Ts,'analytic:',Ta); assert abs(Ts-Ta)<.2
plt.plot(rr,Tk); plt.axvline(rs,ls='--'); plt.xlabel('Radius [m]'); plt.ylabel('Temperature [degC]'); plt.grid(); plt.show()

## B. Gmsh → FEniCSx → PyVista — damaged insulation

Gmsh now becomes valuable because geometry is non-uniform. A 0.4 m local section physically loses half of the insulation thickness. Gmsh creates physical groups for steel, insulation, inner wall and exposed seawater boundary. DOLFINx imports those tags directly and solves the axisymmetric heat equation; PyVista renders the result.

In [ ]:
L=2.; a0,a1=.8,1.2; loss=.5*ti
gmsh.initialize(); gmsh.option.setNumber('General.Terminal',0); gmsh.model.add('damaged_pipe'); o=gmsh.model.occ; st=o.addRectangle(0,ri,0,L,ts); ins=o.addRectangle(0,rs,0,L,ti); notch=o.addRectangle(a0,ro-loss,0,a1-a0,loss); ic,_=o.cut([(2,ins)],[(2,notch)],removeObject=True,removeTool=True); o.fragment([(2,st)],ic); o.synchronize(); surf=[t for d,t in gmsh.model.getEntities(2)]; steel=[]; insulation=[]
for s in surf:
 zc,rc,_=o.getCenterOfMass(2,s); (steel if rc<rs else insulation).append(s)
gmsh.model.addPhysicalGroup(2,steel,1); gmsh.model.setPhysicalName(2,1,'steel'); gmsh.model.addPhysicalGroup(2,insulation,2); gmsh.model.setPhysicalName(2,2,'insulation'); bd=gmsh.model.getBoundary([(2,s) for s in surf],combined=True,oriented=False); inn=[]; out=[]; ends=[]
for d,c in bd:
 zc,rc,_=o.getCenterOfMass(d,c)
 if np.isclose(rc,ri,atol=1e-7): inn.append(c)
 elif np.isclose(zc,0,atol=1e-7) or np.isclose(zc,L,atol=1e-7): ends.append(c)
 else: out.append(c)
gmsh.model.addPhysicalGroup(1,inn,11); gmsh.model.addPhysicalGroup(1,out,12);
if ends: gmsh.model.addPhysicalGroup(1,ends,13)
gmsh.option.setNumber('Mesh.CharacteristicLengthMin',.012); gmsh.option.setNumber('Mesh.CharacteristicLengthMax',.04); gmsh.model.mesh.generate(2); data=gmshio.model_to_mesh(gmsh.model,MPI.COMM_WORLD,0,gdim=2); gmsh.finalize(); gm=data.mesh if hasattr(data,'mesh') else data[0]; ct=data.cell_tags if hasattr(data,'cell_tags') else data[1]; ft=data.facet_tags if hasattr(data,'facet_tags') else data[2]
V=fem.functionspace(gm,('Lagrange',1)); u=ufl.TrialFunction(V); v=ufl.TestFunction(V); X=ufl.SpatialCoordinate(gm); r=X[1]; dx=ufl.Measure('dx',domain=gm,subdomain_data=ct); ds=ufl.Measure('ds',domain=gm,subdomain_data=ft); aa=ks*ufl.inner(ufl.grad(u),ufl.grad(v))*2*np.pi*r*dx(1)+ki*ufl.inner(ufl.grad(u),ufl.grad(v))*2*np.pi*r*dx(2)+hi*u*v*2*np.pi*r*ds(11)+ho*u*v*2*np.pi*r*ds(12); ll=hi*(Tg+273.15)*v*2*np.pi*r*ds(11)+ho*(Tsea+273.15)*v*2*np.pi*r*ds(12); uh=LinearProblem(aa,ll,petsc_options_prefix='gm_',petsc_options={'ksp_type':'preonly','pc_type':'lu'}).solve(); xyz=V.tabulate_dof_coordinates(); inner=(uh.x.array-273.15)[np.isclose(xyz[:,1],ri)]; print('Local min inner-wall T:',float(inner.min())); assert uh.x.array.min()>=Tsea+273.15-1e-6 and uh.x.array.max()<=Tg+273.15+1e-6
cells,types,pts=plot.vtk_mesh(V); grid=pv.UnstructuredGrid(cells,types,pts); grid.point_data['Temperature [degC]']=uh.x.array-273.15; pl=pv.Plotter(off_screen=True,window_size=(1000,420)); pl.add_mesh(grid,scalars='Temperature [degC]',show_edges=True); pl.view_xy(); pl.screenshot('/tmp/gmsh_stack.png'); from IPython.display import Image,display; display(Image('/tmp/gmsh_stack.png'))

## C. NeqSim diffusion → porous-rock FEM

NeqSim provides the multicomponent molecular diffusion coefficient. A porous-medium screen applies $D_{rock}=\phi D_{mol}/\tau$ before solving transient 2D diffusion with scikit-fem. Porosity/tortuosity are rock-model inputs, not thermodynamic properties.

In [ ]:
phi,tau=.22,2.5; Dmol=pr['D_CO2']; Dr=phi/tau*Dmol; print('Dmol=',Dmol,'Drock=',Dr)
m2=MeshTri.init_tensor(np.linspace(0,2,61),np.linspace(0,1,31)); b2=Basis(m2,ElementTriP1())
@BilinearForm
def mass(u,v,w): return u*v
@BilinearForm
def dif(u,v,w): return Dr*dot(grad(u),grad(v))
M=asm(mass,b2); K=asm(dif,b2); dt=6*3600.; left=b2.get_dofs(lambda x:np.isclose(x[0],0)).all(); cc=np.zeros(b2.N); snaps={}
for n in range(80):
 xbc=np.zeros(b2.N); xbc[left]=1.; Ac,bc,x0,I=condense(M+dt*K,M@cc,x=xbc,D=left); cc=x0.copy(); cc[I]=solve(Ac,bc)
 if n in (9,39,79): snaps[(n+1)*dt/86400]=cc.copy()
xy=m2.p; mid=np.where(np.isclose(xy[1],.5,atol=.02))[0]; order=np.argsort(xy[0,mid])
for d,c in snaps.items(): plt.plot(xy[0,mid][order],c[mid][order],label=f'{d:.1f} d')
plt.xlabel('Distance [m]'); plt.ylabel('Normalized CO2'); plt.legend(); plt.grid(); plt.show(); assert cc.min()>=-1e-9 and cc.max()<=1+1e-9

## D. FEniCSx wellbore-to-formation heat conduction

A 200 m well segment is embedded in a rock domain with geothermal far-field temperature. The well boundary uses a convection coefficient derived from the same NeqSim fluid. This demonstrates why FEniCSx is useful when the result is a 2D field rather than a scalar outlet temperature.

In [ ]:
H,rw,rf=200.,.10,15.; kr=2.5; hw=min(hi,1500.); Tfluid=80.; Tfar0=25.; gradT=.03
wm=mesh.create_rectangle(MPI.COMM_WORLD,np.array([[0.,rw],[H,rf]]),[70,40],cell_type=mesh.CellType.triangle); fd=wm.topology.dim-1; wi=mesh.locate_entities_boundary(wm,fd,lambda x:np.isclose(x[1],rw)); fa=mesh.locate_entities_boundary(wm,fd,lambda x:np.isclose(x[1],rf)); en=np.hstack([wi,fa]).astype(np.int32); va=np.hstack([np.ones(len(wi),np.int32),2*np.ones(len(fa),np.int32)]); o=np.argsort(en); tag=mesh.meshtags(wm,fd,en[o],va[o]); Vw=fem.functionspace(wm,('Lagrange',1)); u=ufl.TrialFunction(Vw); v=ufl.TestFunction(Vw); X=ufl.SpatialCoordinate(wm); z=X[0]; r=X[1]; ds=ufl.Measure('ds',domain=wm,subdomain_data=tag); dx=ufl.Measure('dx',domain=wm); Tf=Tfar0+273.15+gradT*z; aa=kr*ufl.inner(ufl.grad(u),ufl.grad(v))*2*np.pi*r*dx+hw*u*v*2*np.pi*r*ds(1)+50*u*v*2*np.pi*r*ds(2); ll=hw*(Tfluid+273.15)*v*2*np.pi*r*ds(1)+50*Tf*v*2*np.pi*r*ds(2); Tw=LinearProblem(aa,ll,petsc_options_prefix='well_',petsc_options={'ksp_type':'preonly','pc_type':'lu'}).solve(); c=Vw.tabulate_dof_coordinates(); wT=(Tw.x.array-273.15)[np.isclose(c[:,1],rw)]; print('Well wall range:',float(wT.min()),float(wT.max()),'degC'); cells,types,pts=plot.vtk_mesh(Vw); gg=pv.UnstructuredGrid(cells,types,pts); gg.point_data['T [degC]']=Tw.x.array-273.15; pl=pv.Plotter(off_screen=True,window_size=(850,500)); pl.add_mesh(gg,scalars='T [degC]'); pl.view_xy(); pl.screenshot('/tmp/well.png'); display(Image('/tmp/well.png'))

## Recommended hierarchy

- **NeqSim**: thermodynamics, process and flow boundary conditions.
- **scikit-fem**: simple/transparent FEM and rapid Colab demonstrations.
- **Gmsh**: geometry, physical groups and reusable meshing.
- **FEniCSx**: detailed heat, mechanics, diffusion and multiphysics PDEs.
- **PyVista**: common visualization of meshes and fields.
- **OpenFOAM**: CFD when momentum/velocity/pressure fields dominate.

The detailed companion notebook extends the pipeline case to shutdown cooldown, hydrate no-touch time and thermo-elastic stress. Natural later extensions are separator/nozzle thermal stress, buried-pipeline soil domains, electrochemistry/corrosion and coupled geomechanics.